# NumCompute quickstart

End-to-end walk through the package: CSV in, preprocessing, statistics, sort/rank, metrics, gradients, pipeline and a benchmark.

In [1]:
import numpy as np
import sys
sys.path.insert(0, "..")  # Find numcompute sample csv in the parent folder.

from numcompute import preprocessing as pp
from numcompute import stats
from numcompute.utils import iter_batches

sys.path.insert(0, "..")  # Find numcompute sample csv in the parent folder.
sample_path = "data/sample.csv"

np.set_printoptions(precision=3, suppress=True)

## 1. Read CSV (`io.py`)

TODO (Jose): replace `np.genfromtxt` with `io.read_csv` once it lands. `sample.csv` has blank cells, which become NaN.

In [11]:
X_num = np.genfromtxt('sample.csv', delimiter=',', skip_header=1, usecols=(0, 1, 2))          # age, income, height; blanks -> NaN
city = np.genfromtxt('sample.csv', delimiter=',', skip_header=1, usecols=3, dtype=str).reshape(-1, 1)
print(X_num)
print(city.ravel())


# Shuhao edit: io file should be able to work following code:
# from numcompute.io import load_csv
# X = load_csv(sample_path, delimiter=",", skip_header=1)

# When finish io, please change all genfrometxt as io.load_csv
X = np.genfromtxt(sample_path, delimiter=',', skip_header=1, dtype=str)


[[    25.    52000.        1.72]
 [    32.         nan      1.8 ]
 [    41.    88000.        1.65]
 [    29.    61000.         nan]
 [    52.   120000.        1.78]
 [    37.    74000.        1.69]
 [    23.    39000.        1.83]
 [    45.         nan      1.71]]
['Adelaide' 'Sydney' 'Melbourne' 'Adelaide' 'Sydney' '' 'Adelaide'
 'Melbourne']


## 2. Preprocess (`preprocessing.py`)

Impute missing numbers, then scale. Fit on train, apply to new rows with the same statistics.

In [ ]:
imp = pp.SimpleImputer(strategy='median')
X_full = imp.fit_transform(X_num)
print('fill values:', imp.statistics_)

scaler = pp.StandardScaler().fit(X_full)
Z = scaler.transform(X_full)
print('mean after scaling:', Z.mean(axis=0), ' std:', Z.std(axis=0))

print(pp.MinMaxScaler((0, 1)).fit_transform(X_full)[:3])

In [ ]:
enc = pp.OneHotEncoder(handle_unknown='ignore').fit(city)
print('categories:', enc.categories_[0])
print(enc.transform(city))          # blank city -> all zeros
print(enc.transform([['Perth']]))    # unseen city, ignored

In [ ]:
# scaler fitted on 3 features rejects a 2-feature matrix with a clear message
try:
    scaler.transform(X_full[:, :2])
except ValueError as e:
    print('ValueError:', e)

## 3. Descriptive statistics (`stats.py`)

`axis=0` summarises each column. `skip_nan=True` ignores NaN; by default NaN propagates like NumPy.

In [ ]:
print('mean   ', stats.mean(X_num, axis=0))
print('mean   ', stats.mean(X_num, axis=0, skip_nan=True), '(skip NaN)')
print('median ', stats.median(X_num, axis=0, skip_nan=True))
print('std    ', stats.std(X_num, axis=0, ddof=1, skip_nan=True))
print('min/max', stats.min(X_num, axis=0, skip_nan=True), stats.max(X_num, axis=0, skip_nan=True))

In [ ]:
print(stats.quantile(X_num, [0.25, 0.5, 0.75], axis=0, skip_nan=True))   # shape (3, n_features)
counts, edges = stats.histogram(X_num[:, 1], bins=4)
print('income histogram:', counts, edges)

In [ ]:
# streaming stats: same numbers as the batch version, O(d) memory
rs = stats.RunningStats(skip_nan=True)
for batch in iter_batches(X_num, 3):
    rs.update(batch)
print('rows seen:', rs.n)
print('running mean:', rs.mean)
print('running std :', rs.std(ddof=1))

## 4. Sort, search, top-k and rank (`sort_search.py`, `rank.py`)

TODO (Faris): top-k on income, quickselect, binary search, rank with ties, percentile.

In [ ]:
# TODO (Faris)

## 5. Metrics (`metrics.py`)

TODO (Jose): accuracy / precision / recall / F1, confusion matrix, mse.

## 6. Pipeline (`pipeline.py`)

A Pipeline runs named steps in order. Here we use the same CSV data loaded as `X`. Each step receives the output of the previous step.

### 6.1 Fill and standardize numerical columns

Use the first 900 rows to fit the pipeline. Fill missing age/income values with 0, then standardize each column. Zero filling is used only as a simple example.

In [19]:
from numcompute.pipeline import Pipeline
from numcompute.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from numcompute.preprocessing import SimpleImputer as Imputer

# numeric_input = X[:, :2].astype(float)
numeric_input = np.genfromtxt(sample_path, delimiter=",", skip_header=1, usecols=(0, 1), dtype=float,)
X_train = numeric_input[:900]
X_test = numeric_input[900:]

numeric_pipe = Pipeline([
    ("impute", Imputer(strategy="constant",fill_value=0)),
    ("scale", StandardScaler()),
])

X_train_ready = numeric_pipe.fit_transform(X_train)
print("Training shape:", X_train_ready.shape)
print(X_train_ready[:5])
print("Training column means:", np.round(X_train_ready.mean(axis=0), 6))

Training shape: (900, 2)
[[-1.209 -1.211]
 [-2.314 -0.972]
 [-0.104 -1.928]
 [-0.656 -0.733]
 [-0.933 -1.091]]
Training column means: [-0.  0.]


### 6.2 Reuse the fitted pipeline

Call `transform` on the remaining 300 rows. This reuses the parameters learned from the first 900 rows and does not fit again.

In [13]:
X_test_ready = numeric_pipe.transform(X_test)
print("Test shape:", X_test_ready.shape)
print(X_test_ready[:5])

Test shape: (300, 2)
[[ 0.     0.626]
 [ 0.     1.649]
 [-1.016 -0.441]
 [ 1.309 -0.249]
 [-0.152  0.   ]]


### 6.3 Fill and encode cities

For the categorical column, replace NaN with `Unknown`, then apply one-hot encoding.

In [24]:
# Shuhao -> Dat: please check this, fill_value for string should be avaliable as well. Thanks.
#                And for Imputer, if I just leave strategy as blank, only enter parameter fill_value, it would auto
#                fill the value with mean but not the actual fill_value. Please take a look at this. :)
test_numeric_pipe1 = Pipeline([
    ("impute", Imputer(strategy="constant",fill_value=0)),
    ("scale", StandardScaler()),
])

test_numeric_pipe2 = Pipeline([
    ("impute", Imputer(fill_value=0)),
    ("scale", StandardScaler()),
])

Test1_out = test_numeric_pipe1.fit_transform(X_train)
Test2_out = test_numeric_pipe2.fit_transform(X_train)

print("Test1_out:\n", Test1_out[:5])
print("===================================================================")
print("Test2_out:\n", Test2_out[:5])


city_pipe = Pipeline([
    ("impute", Imputer(fill_value="Unknown")),
    ("encode", OneHotEncoder()),
])

cities_ready = city_pipe.fit_transform(X[:, 2:3])
print("Encoded city shape:", cities_ready.shape)
print(cities_ready[:5])

Test1_out:
 [[-1.209 -1.211]
 [-2.314 -0.972]
 [-0.104 -1.928]
 [-0.656 -0.733]
 [-0.933 -1.091]]
Test2_out:
 [[-1.614 -1.614]
 [ 0.    -1.331]
 [-0.285  0.   ]
 [-0.949 -1.047]
 [-1.282 -1.473]]


ValueError: could not convert string to float: 'Unknown'

## 7. Finite-Difference Gradients and Jacobians

Estimate derivatives by slightly changing each input coordinate. Forward differences compare `f(x+h)` with `f(x)`; central differences compare `f(x+h)` with `f(x-h)`.

Use the first standardized numerical row from the CSV pipeline as the evaluation point. This keeps age and income on comparable scales.

In [25]:
from numcompute.optim import grad, jacobian

x = X_train_ready[0].copy()
print("Evaluation point:", x)

Evaluation point: [-1.209 -1.211]


### 7.1 Gradient of a scalar function

For `f(x) = sum(x**2)`, the analytical gradient is `2*x`. The result has the same shape as the one-dimensional input.

In [26]:
def f(x):
    return np.sum(x ** 2)

central_gradient = grad(f, x, h=1e-5, method="central")
forward_gradient = grad(f, x, h=1e-5, method="forward")
exact_gradient = 2 * x

print("Central gradient:", central_gradient)
print("Forward gradient:", forward_gradient)
print("Analytical gradient:", exact_gradient)
print("Central result matches:", np.allclose(central_gradient, exact_gradient))

Central gradient: [-2.418 -2.422]
Forward gradient: [-2.418 -2.422]
Analytical gradient: [-2.418 -2.422]
Central result matches: True


### 7.2 Jacobian of a vector function

For `F(x) = [x[0]**2 + x[1], x[0]*x[1]]`, the analytical Jacobian is `[[2*x[0], 1], [x[1], x[0]]]`. Rows correspond to output components and columns to input coordinates.

In [27]:
def F(x):
    return np.array([
        x[0] ** 2 + x[1],
        x[0] * x[1],
    ])

central_J = jacobian(F, x, method="central")
forward_J = jacobian(F, x, method="forward")
exact_J = np.array([[2 * x[0], 1], [x[1], x[0]]])

print("Central Jacobian:")
print(central_J)
print("Forward Jacobian:")
print(forward_J)
print("Analytical Jacobian:")
print(exact_J)
print("Shape:", central_J.shape)
print("Central result matches:", np.allclose(central_J, exact_J))

Central Jacobian:
[[-2.418  1.   ]
 [-1.211 -1.209]]
Forward Jacobian:
[[-2.418  1.   ]
 [-1.211 -1.209]]
Analytical Jacobian:
[[-2.418  1.   ]
 [-1.211 -1.209]]
Shape: (2, 2)
Central result matches: True


## 8. Benchmark (`benchmarking.py`)

TODO (Jose): vectorised vs loop timings; paste the table into README.

In [ ]:
# TODO (Jose)